# 3DEP Extras — QC Dashboard

Generic terrain QC checks across AOIs.

In [ ]:
import ee, numpy as np, pandas as pd

ee.Authenticate()
ee.Initialize(project="shrubwise-dc-488219")
THREEDEP_10M = ee.ImageCollection("USGS/3DEP/10m_collection")
AOI_BLISS = ee.Geometry.Rectangle([-120.1018846, 38.99274873, -120.0899834, 39.0020357], geodesic=False)


In [ ]:
def threedep_dem() -> ee.Image:
    return THREEDEP_10M.mosaic().select("elevation").toFloat().rename("elevation")

def qc_range_coverage(image: ee.Image, band: str, aois: dict, *, scale=10):
    img = image.select(band).toFloat()
    rows = []
    for name, geom in aois.items():
        cnt = img.reduceRegion(ee.Reducer.count(), geom, scale, maxPixels=1e9, bestEffort=True).get(band)
        rows.append({"aoi": name, "valid_count": ee.Number(cnt).getInfo() if cnt is not None else 0})
    return pd.DataFrame(rows).set_index("aoi")

def qc_distribution(image: ee.Image, band: str, aois: dict, *, scale=10):
    img = image.select(band).toFloat()
    reducer = (ee.Reducer.mean()
               .combine(ee.Reducer.median(), sharedInputs=True)
               .combine(ee.Reducer.min(), sharedInputs=True)
               .combine(ee.Reducer.max(), sharedInputs=True)
               .combine(ee.Reducer.percentile([5,25,75,95]), sharedInputs=True))
    rows = []
    for name, geom in aois.items():
        stats = img.reduceRegion(reducer, geom, scale, maxPixels=1e9, bestEffort=True).getInfo()
        rows.append({"aoi": name, **stats})
    return pd.DataFrame(rows).set_index("aoi")

dem = threedep_dem()
AOIS = {"DL_Bliss": AOI_BLISS}
print(qc_range_coverage(dem, "elevation", AOIS))
print(qc_distribution(dem, "elevation", AOIS))
